In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

In [2]:
url = "https://cozoni.us/collections"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)

with open("cozoni_homepage.html", "w", encoding="utf-8") as file:
    file.write(response.text)
print (response)

<Response [200]>


In [3]:
with open("cozoni_homepage.html", "r", encoding="utf-8") as file:
    local_html = file.read()

soup = BeautifulSoup(local_html, "html.parser")

# Search collection folders
category_links = soup.find_all("a", href=True)
cozoni_collections = [link['href'] for link in category_links if "/collections/" in link['href']]

# Remove duplicates
unique_collections = list(set(cozoni_collections))

print(f"Found {len(unique_collections)} collection.")

Found 28 collection.


In [4]:
discovery_url = "https://cozoni.us/collections.json"

json_response = requests.get(discovery_url, headers=headers)
collections_data = json_response.json().get("collections", [])


discovered_menu = []
for folder in collections_data:
    discovered_menu.append({
        "Clean Category Title": folder["title"],
        "URL Slug (Handle)": folder["handle"],
        "Total Products in Database": folder.get("products_count", 0)
    })


discovery_df = pd.DataFrame(discovered_menu)

print(f"Backend Status {json_response.status_code}")
print(f"Found {len(discovery_df)} categories inside the database architecture.\n")
discovery_df

Backend Status 200
Found 30 categories inside the database architecture.



,Clean Category Title,URL Slug (Handle),Total Products in Database
0,ACCESSORIES,accessories,10
1,ARMCHAIRS,armchairs,26
2,BAR STOOLS,bar-stools,12
3,BEST SELLERS,best-sellers,145
4,CHAIRS,chairs,94
5,COFFEE TABLES,coffee-tables,17
6,CONSOLE TABLES,console-tables,1
7,DESKS,desks,11
8,DINING,dining,79
9,DINING CHAIRS,dining-chairs,56


In [5]:
base_api_url = "https://cozoni.us/collections"
raw_data_pool = []


categories = [
    "accessories", "armchairs", "bar-stools", "best-sellers", "chairs",
    "coffee-tables", "console-tables", "desks", "dining", "dining-chairs",
    "dining-stools", "dining-tables", "drawers-trolleys", "folding-chairs",
    "furnishing-accessories", "kids", "living", "lounge-chairs", "mirrors",
    "new", "office-chairs", "office-tables", "ottomans", "outdoor",
    "rugs", "sales", "shelving-units", "side-tables", "sideboards", "sofa"
]

print(f"Scanning {len(categories)} departments...")

for cat in categories:
    page = 1
    dept = cat.replace("-", " ").title()

    while True:
        url = f"{base_api_url}/{cat}/products.json?page={page}"
        res = requests.get(url, headers=headers)

        if res.status_code != 200:
            break

        products = res.json().get("products", [])
        if not products:
            break

        for item in products:
            raw_data_pool.append({
                "Department": dept,
                "Product Title": item["title"],
                "Raw Price": item["variants"][0]["price"]
            })

        time.sleep(random.uniform(0.3, 0.7))
        page += 1

print(f"Done. {len(raw_data_pool)} raw entries.")

Scanning 30 departments...
Done. 910 raw entries.


In [6]:

df_raw_view = pd.DataFrame(raw_data_pool)

print(f" DATA GRID PROFILE: {df_raw_view.shape[0]} rows by {df_raw_view.shape[1]} columns\n")
df_raw_view

 DATA GRID PROFILE: 910 rows by 3 columns



,Department,Product Title,Raw Price
0,Accessories,COZONI Molly Cat House,159.00
1,Accessories,COZONI Magazine Rack,128.00
2,Accessories,COZONI Cheese Floor Mirror,728.00
3,Accessories,COZONI Wavy Floor Mirror,799.00
4,Accessories,COZONI Folding Step Ladder,199.00
...,...,...,...
905,Sideboards,COZONI Egbert Sideboard - 6 Drawers,2199.00
906,Sideboards,COZONI Egbert Sideboard,2199.00
907,Sideboards,COZONI Mateo Sideboard,2499.00
908,Sideboards,COZONI Zion 2 Door Sideboard,2580.00


In [7]:
df_raw_view.to_csv("cozoni_raw_catalog_view.csv", index=False)


In [9]:
df_transform = df_raw_view.copy()

df_transform["Price (USD)"] = df_transform["Raw Price"].astype(float)

df_transformed_clean = df_transform.drop_duplicates(subset=["Product Title"], keep="first").copy()

def generate_perfect_sku(title):
    clean_title = title.replace("COZONI | ", "").replace("COZONI ", "").strip()
    return f"INF-COZ-{clean_title[:3].upper()}-2026"

df_transformed_clean["Internal SKU"] = df_transformed_clean["Product Title"].apply(generate_perfect_sku)

final_master_catalog = df_transformed_clean[["Internal SKU", "Department", "Product Title", "Price (USD)"]]

final_master_catalog.to_csv("cozoni_clean_catalog_view.csv", index=False)

print(f"Starting Raw Data: {len(df_transform)}")
print(f"Transformed Unique Rows: {len(final_master_catalog)}")
print(f"Duplicate Rows Deleted: {len(df_transform) - len(final_master_catalog)}")

final_master_catalog.head(5)

Starting Raw Data: 910
Transformed Unique Rows: 214
Duplicate Rows Deleted: 696


,Internal SKU,Department,Product Title,Price (USD)
0,INF-COZ-MOL-2026,Accessories,COZONI Molly Cat House,159.0
1,INF-COZ-MAG-2026,Accessories,COZONI Magazine Rack,128.0
2,INF-COZ-CHE-2026,Accessories,COZONI Cheese Floor Mirror,728.0
3,INF-COZ-WAV-2026,Accessories,COZONI Wavy Floor Mirror,799.0
4,INF-COZ-FOL-2026,Accessories,COZONI Folding Step Ladder,199.0
